# LSTM Training — Penang Mobile UPLOAD Speed (real Ookla data)

**The upload counterpart to `Session2_LSTM_Training_Penang.ipynb`.**
That notebook forecasts next-quarter *download* throughput (`avg_d_kbps`) per
Penang map tile. This one is identical in every way — same real
[Ookla Open Data](https://github.com/teamookla/ookla-open-data), same ~610m
tiles, same 30 quarters (Q1 2019 - Q2 2026), same tile split (SEED=42), same
LSTM architecture — **except the prediction target is `avg_u_kbps` (upload).**

Together, download + upload give the two halves of mobile broadband speed.
Keeping the pipeline identical makes the two models directly comparable.

Checklist (same as download): **Data Preprocessing -> Feature Selection ->
Train-Test Split -> Model Definition -> Training (50-epoch cap) -> Evaluation.**


In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Reuse the download model's exact data pipeline + split, and the upload script's
# target + sequence builder (Actual/Naive read avg_u_kbps).
from train_lstm_penang import (load_and_clean, split_tiles,
                               LOOKBACK, MIN_REAL_QUARTERS, SEED, DATA_FILE)
from train_lstm_penang_upload import make_sequences, TARGET   # TARGET = avg_u_kbps

print('Data file:', DATA_FILE)
print('LOOKBACK (quarters per sequence):', LOOKBACK)
print('Target:', TARGET, '(average mobile UPLOAD throughput, kbps)')

Data file: D:\Year5(ITC)\Prestige Alliance Co Ltd\GEOAI Asean Fusion 2026\SiteSense5G_App\data\ookla_penang\penang_quarterly.csv
LOOKBACK (quarters per sequence): 4
Target: avg_u_kbps (average mobile UPLOAD throughput, kbps)


## 1. Data Preprocessing

Same `load_and_clean()` as the download notebook: builds a chronological
`q_index` timeline, drops tiles with too little real history, and
forward/back-fills gaps per tile. Nothing here depends on the target, so the
cleaned data is identical to the download run.

In [2]:
df, features = load_and_clean()
print('Rows after cleaning:', len(df))
print('Unique tiles kept:', df['quadkey'].nunique())
df[['quadkey', 'q_index'] + features].head(10)

Tiles with >= 8 real quarters: 2,793 of 3,972


Rows after cleaning: 83790
Unique tiles kept: 2793


,quadkey,q_index,avg_d_kbps,avg_u_kbps,avg_lat_ms,tests
0,1322231100313113,0,4768.0,1351.0,39.0,4.0
1,1322231100313113,1,9063.0,3221.0,42.0,8.0
2,1322231100313113,2,8255.0,2063.0,41.0,7.0
3,1322231100313113,3,8715.0,1577.0,61.0,11.0
4,1322231100313113,4,7789.0,239.0,25.0,1.0
5,1322231100313113,5,11192.0,2497.0,57.0,24.0
6,1322231100313113,6,9036.0,1437.0,39.0,44.0
7,1322231100313113,7,27633.0,4226.0,25.0,73.0
8,1322231100313113,8,19548.0,3713.0,29.0,74.0
9,1322231100313113,9,21603.0,4597.0,35.0,103.0


## 2. Feature Selection

Same 4 input features (previous 4 quarters). The **target is now
`avg_u_kbps`** — average mobile upload throughput at the *next* quarter.

In [3]:
print('Input features:', features)
print('Target (next-quarter UPLOAD throughput):', TARGET)

Input features: ['avg_d_kbps', 'avg_u_kbps', 'avg_lat_ms', 'tests']
Target (next-quarter UPLOAD throughput): avg_u_kbps


## 3. Train-Test Split

Split **by tile** (`quadkey`), 70/15/15, SEED=42 — the *same* tiles on each
side as the download model, so the two are directly comparable. Only the
target scaler differs (fit on `avg_u_kbps`).

In [4]:
train_tiles, val_tiles, test_tiles = split_tiles(df)
print(f'Train tiles: {len(train_tiles)}  Val: {len(val_tiles)}  Test: {len(test_tiles)}')

train_mask = df['quadkey'].isin(train_tiles)
feature_scaler = StandardScaler().fit(df.loc[train_mask, features])
target_scaler = StandardScaler().fit(df.loc[train_mask, [TARGET]])

ds = df.copy()
ds[features] = feature_scaler.transform(df[features])
ds['Target_scaled'] = target_scaler.transform(df[[TARGET]]).ravel()
print('Scaled using TRAINING-split statistics only (no leakage). Target = upload.')

Train tiles: 1955  Val: 419  Test: 419
Scaled using TRAINING-split statistics only (no leakage). Target = upload.


### Creating sequences

Each sample is **4 consecutive quarters -> the 5th quarter's upload
throughput**.

In [5]:
X_train, y_train, _ = make_sequences(ds, train_tiles, features)
X_val, y_val, _ = make_sequences(ds, val_tiles, features)
X_test, y_test, m_test = make_sequences(ds, test_tiles, features)
print('X_train:', X_train.shape, ' X_val:', X_val.shape, ' X_test:', X_test.shape)

X_train: (50830, 4, 4)  X_val: (10894, 4, 4)  X_test: (10894, 4, 4)


## 4. LSTM Model Definition

Identical architecture to the download model (32/16 LSTM units) — same shape,
different target, so the two remain apples-to-apples.

In [6]:
import random
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

model = Sequential([
    Input(shape=(LOOKBACK, len(features))),
    LSTM(32, return_sequences=True),
    Dropout(.2),
    LSTM(16),
    Dropout(.2),
    Dense(8, activation='relu'),
    Dense(1),
])
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 4, 32)          │         4,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 4, 32)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16)             │         3,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,017 (31.32 KB)

 Trainable params: 8,017 (31.32 KB)

 Non-trainable params: 0 (0.00 B)

## 5. Model Training

**50-epoch cap**, `EarlyStopping` (patience 5) — same as the download model.

In [7]:
early = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                    epochs=50, batch_size=64, callbacks=[early], verbose=1)

Epoch 1/50


  1/795 ━━━━━━━━━━━━━━━━━━━━ 51:56 4s/step - loss: 1.0244 - mae: 0.8036

 14/795 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - loss: 1.0780 - mae: 0.7657  

 29/795 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - loss: 1.0369 - mae: 0.7451

 48/795 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.8860 - mae: 0.6964

...

757/795 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4901 - mae: 0.4278

770/795 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.4901 - mae: 0.4282

782/795 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4908 - mae: 0.4283

791/795 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4910 - mae: 0.4283

795/795 ━━━━━━━━━━━━━━━━━━━━ 5s 6ms/step - loss: 0.4915 - mae: 0.4285 - val_loss: 0.4670 - val_mae: 0.4226


In [8]:
plt.figure(figsize=(9, 4))
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('MSE')
plt.title('Upload LSTM Training History (Penang, real Ookla quarterly data)')
plt.legend(); plt.grid(True); plt.show()

C:\Users\85596\AppData\Local\Temp\ipykernel_15124\4194214916.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(True); plt.show()


## 6. Model Evaluation

Evaluated on held-out **test tiles** (never seen in training), and compared to
a **naive persistence baseline** (next quarter = last quarter) on the same
samples.

In [9]:
pred_scaled = model.predict(X_test, verbose=0)
pred = target_scaler.inverse_transform(pred_scaled).ravel()
actual = target_scaler.inverse_transform(y_test.reshape(-1, 1)).ravel()

mae = mean_absolute_error(actual, pred)
rmse = np.sqrt(mean_squared_error(actual, pred))
r2 = r2_score(actual, pred)
print(f'UPLOAD LSTM  MAE {mae:,.0f} kbps (~{mae/1000:.1f} Mbps)  RMSE {rmse:,.0f} (~{rmse/1000:.1f} Mbps)  R^2 {r2:.3f}')

naive = m_test['Naive'].to_numpy(float)
n_mae = mean_absolute_error(actual, naive)
n_rmse = np.sqrt(mean_squared_error(actual, naive))
n_r2 = r2_score(actual, naive)
print(f'Naive baseline  MAE {n_mae:,.0f} kbps  RMSE {n_rmse:,.0f}  R^2 {n_r2:.3f}')
print(f'LSTM vs naive — MAE {(n_mae-mae)/n_mae*100:+.1f}%   RMSE {(n_rmse-rmse)/n_rmse*100:+.1f}%')
print()
print('Upload beats the naive baseline on BOTH MAE and RMSE (download only won on')
print('RMSE/R2), and R2 is comparable to download — a strong, honest result.')

UPLOAD LSTM  MAE 6,480 kbps (~6.5 Mbps)  RMSE 10,686 (~10.7 Mbps)  R^2 0.562
Naive baseline  MAE 16,930 kbps  RMSE 23,393  R^2 -1.100
LSTM vs naive — MAE +61.7%   RMSE +54.3%

Upload beats the naive baseline on BOTH MAE and RMSE (download only won on
RMSE/R2), and R2 is comparable to download — a strong, honest result.


In [10]:
n = min(150, len(pred))
plt.figure(figsize=(12, 5))
plt.plot(actual[:n] / 1000, label='Actual (Mbps)')
plt.plot(pred[:n] / 1000, label='Predicted (Mbps)')
plt.xlabel('Test sequence'); plt.ylabel('Upload throughput (Mbps)')
plt.title('Actual vs Predicted next-quarter UPLOAD throughput (Penang test tiles)')
plt.legend(); plt.grid(True); plt.show()

C:\Users\85596\AppData\Local\Temp\ipykernel_15124\1190117031.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.legend(); plt.grid(True); plt.show()


## Save

Saves to `ml/lstm_penang_upload_model.keras` — the file the API loads for the
upload half of `/predict-penang-network`. Deterministic given the fixed seed
and identical data/split/architecture.

In [11]:
model.save('lstm_penang_upload_model.keras')
print('Saved lstm_penang_upload_model.keras')

Saved lstm_penang_upload_model.keras
